# v0.5 - Distillation pour le sentiment (IMDB), T4 16 Go

**Teacher** : `Mistral-7B-Instruct-v0.3` + un adapter de sentiment (NF4)
**Élève** : un petit modèle (TinyLlama par défaut), entraîné en LoRA

Notebook **reprenable** : tout est écrit sur disque, relancer toutes les cellules saute ce qui est fait.

## Pourquoi le teacher est ré-entraîné ici

Le diagnostic a montré que l'adapter `Falcot922/mistral-nf4-sentiment` répond « negative » à **toutes** les critiques (il a vu
des exemples d'une seule classe : IMDB est trié par label sur le Hub, `train[:N]` sans mélange ne contient que des négatifs).
Distiller un teacher qui répond toujours pareil n'a aucun sens. `TEACHER_MODE` (§1) propose donc :

- `"retrain"` *(défaut)* : ré-entraîne un adapter QLoRA sur des données **mélangées et équilibrées** (~100 steps) : c'est ta v0.3 corrigée ;
- `"base"` : Mistral-Instruct sans adapter (zero-shot, ~92 % chez toi) ;
- `"hub"` : ton ancien adapter (déconseillé : dégénéré, le notebook s'arrête si c'est le cas).

## Principe de la distillation (celui de ta roadmap)

Pour une critique, le teacher donne un score pour chaque classe au premier token de réponse. On garde ces **logits** (soft labels)
et l'élève apprend à les reproduire avec une **KL à température** (+ une cross-entropy sur le vrai label, pondérée par `1 - KD_ALPHA`).
Comme on compare des **classes** (et non le vocabulaire entier), les tokenizers du teacher et de l'élève peuvent différer.
Chaque classe regroupe ses variantes (`positive` / `Positive`) par `logsumexp` : le modèle répond `Positive` avec une majuscule
quand il n'a pas d'adapter, et une lecture sur une seule variante serait fausse.

## Tâches couvertes

| # | Tâche | Où |
|---|-------|----|
| 1 | Teacher Mistral (NF4 + adapter valide), gelé, soft labels sur le train | §4-§5 |
| 2 | Élève entraîné avec **KL + température** et cross-entropy | §6 |
| 3 | Ablation : élève entraîné sur labels durs seuls (`alpha = 0`) | §6 |
| 4 | Tableau teacher / élève brut / élève dur / élève distillé : accuracy **avec intervalle de confiance**, fidélité au teacher, latence, VRAM | §7 |
| 5 | Sauvegarde HF (optionnel) | §8 |

In [ ]:
# L'image Kaggle n'embarque pas bitsandbytes : sans lui le chargement NF4 du teacher leve une ImportError
!pip install -q -U "bitsandbytes>=0.46.1"
# peft refuse de poser un LoRA si un torchao trop ancien est installe (0.10 sur Kaggle) ; on ne s'en sert pas
!pip uninstall -y -q torchao

## 1. Configuration

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # avant d'importer torch
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---------- Teacher ----------
TEACHER_MODE = "retrain"      # "retrain" | "base" | "hub"
# copie NF4 en acces libre du modele officiel : 4 Go au lieu de 14,5, et pas de token HF a fournir
TEACHER_ID   = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
ADAPTER_PATH = "Falcot922/mistral-nf4-sentiment"      # utilise seulement si TEACHER_MODE = "hub" (degenere)

WORK = f"/kaggle/working/v05_distillation_sentiment_{TEACHER_MODE}"   # un dossier par mode : pas de melange
os.makedirs(WORK, exist_ok=True)
TEACHER_ADAPTER_DIR = f"{WORK}/teacher_adapter"

# ---------- Eleve ----------
STUDENT_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"      # tokenizer different du teacher : OK ici

# ---------- Donnees ----------
DATASET_NAME   = "stanfordnlp/imdb"    # "imdb" seul est refuse par les versions recentes de huggingface_hub
TEXT_FIELD     = "text"
LABEL_FIELD    = "label"               # 1 = positive, 0 = negative
N_TRAIN        = 1000                  # exemples pour la distillation
N_TEACHER_TRAIN = 800                  # exemples (DISJOINTS) pour ré-entrainer le teacher
N_EVAL         = 1000                  # exemples d'evaluation (jamais vus)
TEXT_MAX_CHARS = 400                   # comme en v0.3
SEED           = 42

# ---------- Prompts et mots-labels (groupes de variantes : index 0 = positive, 1 = negative) ----------
TEACHER_PROMPT = ("[INST] Analyze the sentiment of this movie review and respond with exactly one word: "
                  "positive or negative.\n\nReview: {text} [/INST]")
TEACHER_LABELS = (("positive", "Positive"), ("negative", "Negative"))

STUDENT_PROMPT = ("Analyze the sentiment of this movie review and respond with exactly one word: "
                  "positive or negative.\n\nReview: {text}\nSentiment:")
STUDENT_LABELS = ((" positive", " Positive"), (" negative", " Negative"))

# ---------- Entrainement du teacher (mode "retrain") ----------
T_EPOCHS, T_BATCH, T_ACCUM, T_LR = 1, 2, 4, 2e-4       # ~100 steps
T_LORA_R, T_LORA_ALPHA = 8, 16                         # comme ta v0.3

# ---------- Distillation ----------
KD_ALPHA = 0.7         # poids de la KL (1 - alpha = cross-entropy sur le vrai label)
KD_TEMP  = 2.0
RUN_HARD_ABLATION = True   # entraine aussi un eleve alpha=0 (labels durs) pour comparer

# ---------- Entrainement eleve ----------
EPOCHS, BATCH, ACCUM, LR = 3, 4, 4, 2e-4
LORA_R, LORA_ALPHA = 16, 32

# ---------- Hugging Face (optionnel) ----------
HF_USER = "Falcot922"

# ---------- Mode mini : verifie en quelques minutes que tout s'enchaine, avant le vrai calcul ----------
MINI = False
if MINI:
    N_TRAIN, N_TEACHER_TRAIN, N_EVAL, EPOCHS = 48, 32, 24, 1

## 2. Imports et utilitaires

In [4]:
import gc, json, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          get_linear_schedule_with_warmup)

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"   [VRAM allouee apres nettoyage : {torch.cuda.memory_allocated()/2**30:.2f} Gio]")

def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def append_jsonl(path, row):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")

def get_tokenizer(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"          # on lit le logit du DERNIER token du prompt
    return tok

BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

# ---------- Tracking pandas ----------
BENCH_CSV = f"{WORK}/benchmark.csv"
BENCH_COLS = ["model", "accuracy", "ci_low", "ci_high", "examples_per_s", "latency_ms",
              "peak_vram_gib", "frac_pred_positive", "n_eval"]

def load_bench():
    return pd.read_csv(BENCH_CSV) if os.path.exists(BENCH_CSV) else pd.DataFrame(columns=BENCH_COLS)

bench_df = load_bench()

def bench_has(name):
    return (bench_df["model"] == name).any()

def bench_record(row):
    global bench_df
    bench_df = pd.concat([bench_df, pd.DataFrame([row], columns=BENCH_COLS)], ignore_index=True)
    bench_df = bench_df.drop_duplicates("model", keep="last")
    bench_df.to_csv(BENCH_CSV, index=False)

print(bench_df)

Empty DataFrame
Columns: [model, accuracy, ci_low, ci_high, examples_per_s, latency_ms, peak_vram_gib, frac_pred_positive, n_eval]
Index: []


## 3. Données (mélangées, classes équilibrées, ensembles disjoints)

In [6]:
imdb = load_dataset(DATASET_NAME)
shuffled_train = imdb["train"].shuffle(seed=SEED)
train_raw         = shuffled_train.select(range(N_TRAIN))                              # distillation
teacher_train_raw = shuffled_train.select(range(N_TRAIN, N_TRAIN + N_TEACHER_TRAIN))   # entrainement du teacher
eval_raw          = imdb["test"].shuffle(seed=SEED).select(range(N_EVAL))              # evaluation

def to_records(raw):
    # c = index de classe : 0 = positive, 1 = negative (meme ordre que TEACHER_LABELS / STUDENT_LABELS)
    return [{"id": i, "text": ex[TEXT_FIELD][:TEXT_MAX_CHARS], "c": 1 - int(ex[LABEL_FIELD])}
            for i, ex in enumerate(raw)]

train_records         = to_records(train_raw)
teacher_train_records = to_records(teacher_train_raw)
eval_records          = to_records(eval_raw)

for name, recs in [("train (distillation)", train_records), ("train (teacher)", teacher_train_records), ("eval", eval_records)]:
    print(f"{name:22s}: {len(recs)} ex., {np.mean([r['c'] == 0 for r in recs]):.0%} positifs")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

train (distillation)  : 1000 ex., 49% positifs
train (teacher)       : 800 ex., 50% positifs
eval                  : 200 ex., 48% positifs


## 4. Lecture des logits de classe, boucle d'entraînement, teacher

- `label_token_ids` : pour chaque classe, le **premier token** de chaque variante (`positive`, `Positive`...) dans le contexte exact du prompt.
- `class_logits` : `logsumexp` des logits des variantes d'une classe -> un score par classe.
- `label_logits` : applique ça au dernier token du prompt (les `position_ids` sont calculés depuis le masque, pour que le padding à gauche ne décale rien).
- `fit` : boucle d'entraînement commune au teacher et aux élèves (fp16 autocast, `GradScaler`, gradient checkpointing, accumulation).

In [8]:
def label_token_ids(tok, template, labels):
    prompt = template.format(text="example")
    p_ids = tok(prompt)["input_ids"]
    groups = []
    for variants in labels:
        g = []
        for w in variants:
            f_ids = tok(prompt + w)["input_ids"]
            assert f_ids[:len(p_ids)] == p_ids, f"Tokenisation instable autour de {w!r}"
            if f_ids[len(p_ids)] not in g:
                g.append(f_ids[len(p_ids)])
        groups.append(g)
    flat = [i for g in groups for i in g]
    assert len(set(flat)) == len(flat), "Une meme token appartient aux deux classes."
    print("   tokens par classe :", [[tok.decode([i]) for i in g] for g in groups])
    return groups

def class_logits(last, groups):
    # last : [B, V] -> [B, nb_classes], logsumexp sur les variantes de chaque classe
    return torch.stack([torch.logsumexp(last[:, g], dim=-1) for g in groups], dim=-1)

@torch.no_grad()
def label_logits(model, tok, prompts, groups, batch_size=8):
    out = []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(model.device)
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        last = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                     position_ids=pos).logits[:, -1, :].float()
        out.append(class_logits(last, groups).cpu())
    return torch.cat(out).numpy()

def fit(model, tok, items, loss_fn, epochs, batch, accum, lr, tag):
    # items : liste de (prompt, payload) ; loss_fn(last_logits [B,V] float32, payloads) -> perte scalaire
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    total_steps = epochs * math.ceil(len(items) / (batch * accum))
    sched = get_linear_schedule_with_warmup(opt, max(1, int(0.05 * total_steps)), total_steps)
    scaler = torch.amp.GradScaler("cuda")
    rng = np.random.default_rng(SEED)
    model.config.use_cache = False
    model.train()

    micro, step, running, last_print = 0, 0, 0.0, 0
    opt.zero_grad()
    for epoch in range(epochs):
        perm = rng.permutation(len(items))
        for b in range(0, len(perm), batch):
            chunk = [items[j] for j in perm[b:b + batch]]
            enc = tok([x[0] for x in chunk], return_tensors="pt", padding=True).to("cuda")
            pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
            with torch.autocast("cuda", dtype=torch.float16):
                last = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                             position_ids=pos).logits[:, -1, :]
            loss = loss_fn(last.float(), [x[1] for x in chunk]) / accum
            scaler.scale(loss).backward()
            running += loss.item() * accum
            micro += 1
            if micro % accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad()
                step += 1
                if step % 10 == 0 or step == total_steps:
                    mean_loss = running / accum / (step - last_print)     # moyenne sur les steps depuis le dernier affichage
                    print(f"   [{tag}] epoch {epoch + 1} | step {step}/{total_steps} | loss {mean_loss:.4f}")
                    append_jsonl(f"{WORK}/loss.jsonl", {"tag": tag, "step": step, "loss": mean_loss})
                    running, last_print = 0.0, step

# ---------- Teacher ----------
def teacher_adapter_path():
    if TEACHER_MODE == "base":
        return None
    if TEACHER_MODE == "hub":
        return ADAPTER_PATH
    return TEACHER_ADAPTER_DIR

def load_teacher_base():
    return AutoModelForCausalLM.from_pretrained(TEACHER_ID, quantization_config=BNB,
                                                device_map={"": 0}, torch_dtype=torch.float16)

def load_teacher():
    m = load_teacher_base()
    path = teacher_adapter_path()
    if path:
        m = PeftModel.from_pretrained(m, path)
    m.eval()
    return m

def train_teacher_adapter():
    # QLoRA sur des donnees melangees et equilibrees. Cible = premier token de la reponse
    # (variante minuscule), cross-entropy sur tout le vocabulaire : l'adapter apprend a repondre
    # "positive"/"negative" correctement, comme en v0.3.
    if os.path.exists(f"{TEACHER_ADAPTER_DIR}/adapter_config.json"):
        print("[teacher] adapter deja entraine, on saute.")
        return
    tok = get_tokenizer(TEACHER_ID)
    groups = label_token_ids(tok, TEACHER_PROMPT, TEACHER_LABELS)
    target_ids = [g[0] for g in groups]
    items = [(TEACHER_PROMPT.format(text=r["text"]), target_ids[r["c"]]) for r in teacher_train_records]
    print(f"[teacher] QLoRA sur {len(items)} exemples equilibres ({T_EPOCHS} epoch)")

    model = load_teacher_base()
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(
        r=T_LORA_R, lora_alpha=T_LORA_ALPHA, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()

    def loss_fn(last, payloads):
        return F.cross_entropy(last, torch.tensor(payloads, device=last.device))

    fit(model, tok, items, loss_fn, T_EPOCHS, T_BATCH, T_ACCUM, T_LR, "teacher")
    model.save_pretrained(TEACHER_ADAPTER_DIR)          # sauvegarde AVANT toute evaluation
    print(f"   adapter teacher sauvegarde -> {TEACHER_ADAPTER_DIR}")

def load_student_fp16(adapter=None):
    m = AutoModelForCausalLM.from_pretrained(STUDENT_ID, torch_dtype=torch.float16, device_map={"": 0})
    if adapter:
        m = PeftModel.from_pretrained(m, adapter).merge_and_unload()
    m.eval()
    return m

if TEACHER_MODE == "retrain":
    train_teacher_adapter()
    cleanup()
else:
    print(f"TEACHER_MODE = {TEACHER_MODE!r} : pas d'entrainement du teacher.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


   tokens par classe : [['pos', 'Pos'], ['negative', 'Neg']]
[teacher] QLoRA sur 800 exemples equilibres (1 epoch)


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

## 5. Étape A : le teacher produit les soft labels

Une seule passe avant par exemple (pas de génération) : c'est rapide. Chaque ligne est écrite tout de suite dans
`teacher_soft.jsonl`, donc la reprise est possible. **Garde-fou** : si le teacher prédit presque toujours la même classe sur
ce jeu équilibré, le notebook **s'arrête** (distiller un teacher dégénéré n'a pas de sens).

In [7]:
TEACHER_JSONL = f"{WORK}/teacher_soft.jsonl"

def step_teacher_soft_labels():
    done = {r["id"] for r in read_jsonl(TEACHER_JSONL)}
    todo = [r for r in train_records if r["id"] not in done]
    if not todo:
        print("[teacher] soft labels deja calcules, on saute.")
        return
    print(f"[teacher] {len(todo)} exemples a traiter ({len(done)} deja faits)")
    tok = get_tokenizer(TEACHER_ID)
    groups = label_token_ids(tok, TEACHER_PROMPT, TEACHER_LABELS)
    model = load_teacher()
    t0 = time.time()
    for i in range(0, len(todo), 8):
        chunk = todo[i:i + 8]
        z = label_logits(model, tok, [TEACHER_PROMPT.format(text=r["text"]) for r in chunk], groups)
        for r, zz in zip(chunk, z):
            append_jsonl(TEACHER_JSONL, {"id": r["id"], "c": r["c"], "z0": float(zz[0]), "z1": float(zz[1])})
        print(f"   {min(i + 8, len(todo))}/{len(todo)} | {(i + len(chunk)) / (time.time() - t0):.1f} ex/s")

step_teacher_soft_labels()
cleanup()

soft = read_jsonl(TEACHER_JSONL)
z = np.array([[r["z0"], r["z1"]] for r in soft])
c = np.array([r["c"] for r in soft])
pred = z.argmax(1)
frac_pos = float(np.mean(pred == 0))
print(f"\n{len(soft)} soft labels | accuracy du teacher sur ces exemples : {(pred == c).mean():.1%}")
print(f"Le teacher predit 'positive' dans {frac_pos:.0%} des cas (vrai taux de positifs : {np.mean(c == 0):.0%})")

if not MINI and (frac_pos > 0.9 or frac_pos < 0.1):
    raise RuntimeError(
        "Teacher degenere : il predit presque toujours la meme classe sur un jeu equilibre. "
        "Utilise TEACHER_MODE = 'retrain' (ou 'base'), supprime le dossier WORK du mode concerne et relance.")

p = torch.softmax(torch.tensor(z), dim=-1)[:, 0].numpy()
print(f"Confiance moyenne du teacher : {np.maximum(p, 1 - p).mean():.1%} "
      "(proche de 100 % = peu d'information dans les soft labels)")

[teacher] 1000 exemples a traiter (0 deja faits)


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

NameError: name 'label_token_ids' is not defined

## 6. Étape B : entraînement de l'élève (LoRA)

Perte = `alpha * KL(softmax(t/T) || softmax(s/T)) * T^2 + (1 - alpha) * cross-entropy(vrai label)`, calculée sur les **scores de classe**
(logsumexp des variantes). L'adapter est sauvegardé **avant** toute évaluation.

- `student_distil` : `alpha = KD_ALPHA` (distillation)
- `student_hard` : `alpha = 0` (même budget, labels durs seulement) : c'est ce qui prouve que la distillation apporte quelque chose.

In [ ]:
def train_student(name, alpha):
    out_dir = f"{WORK}/{name}_adapter"
    if os.path.exists(f"{out_dir}/adapter_config.json"):
        print(f"[{name}] adapter deja present, on saute l'entrainement.")
        return

    tok = get_tokenizer(STUDENT_ID)
    groups = label_token_ids(tok, STUDENT_PROMPT, STUDENT_LABELS)
    soft_by_id = {r["id"]: r for r in read_jsonl(TEACHER_JSONL)}
    items = [(STUDENT_PROMPT.format(text=r["text"]),
              (np.array([soft_by_id[r["id"]]["z0"], soft_by_id[r["id"]]["z1"]], dtype=np.float32), r["c"]))
             for r in train_records if r["id"] in soft_by_id]
    assert items, "Aucun soft label : lance d'abord la cellule du §5."
    print(f"[{name}] alpha={alpha} | {len(items)} exemples | {EPOCHS} epochs")

    model = AutoModelForCausalLM.from_pretrained(STUDENT_ID, torch_dtype=torch.float16, device_map={"": 0})
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()

    def loss_fn(last, payloads):
        s = class_logits(last, groups)
        t = torch.tensor(np.stack([p[0] for p in payloads]), device=s.device)
        y = torch.tensor([p[1] for p in payloads], device=s.device)
        ce = F.cross_entropy(s, y)
        kd = F.kl_div(F.log_softmax(s / KD_TEMP, -1), F.softmax(t / KD_TEMP, -1),
                      reduction="batchmean") * (KD_TEMP ** 2)
        return alpha * kd + (1 - alpha) * ce

    fit(model, tok, items, loss_fn, EPOCHS, BATCH, ACCUM, LR, name)
    model.save_pretrained(out_dir)      # sauvegarde AVANT toute evaluation
    print(f"   adapter sauvegarde -> {out_dir}")

train_student("student_distil", KD_ALPHA)
cleanup()
if RUN_HARD_ABLATION:
    train_student("student_hard", 0.0)
    cleanup()

## 7. Benchmark : teacher / élève brut / élève dur / élève distillé

Tous les modèles sont évalués **de la même façon** : argmax des scores de classe sur le même jeu d'éval fixé
(pas de génération, donc pas de problème de parsing). Le teacher reste en NF4 (comme déployé), les élèves sont en fp16 avec
l'adapter fusionné. Mesures : accuracy avec **intervalle de confiance de Wilson 95 %**, fidélité au teacher, latence batch 1, pic VRAM.

In [ ]:
def run_benchmark(name, tok_id, template, labels, loader):
    if bench_has(name):
        print(f"[bench] {name} deja mesure, on saute.")
        return
    print(f"[bench] {name}")
    cleanup()
    torch.cuda.reset_peak_memory_stats()
    tok = get_tokenizer(tok_id)
    groups = label_token_ids(tok, template, labels)
    model = loader()
    prompts = [template.format(text=r["text"]) for r in eval_records]
    c_true = np.array([r["c"] for r in eval_records])

    label_logits(model, tok, prompts[:1], groups, batch_size=1)          # warmup
    zs, elapsed = [], 0.0
    for p in prompts:                                                      # batch 1 : latence reelle
        torch.cuda.synchronize()
        t0 = time.time()
        zs.append(label_logits(model, tok, [p], groups, batch_size=1)[0])
        torch.cuda.synchronize()
        elapsed += time.time() - t0
    z = np.stack(zs)
    pred = z.argmax(1)

    k, n = int((pred == c_true).sum()), len(c_true)
    lo, hi = wilson(k, n)
    peak = torch.cuda.max_memory_allocated() / 2**30
    json.dump(pred.tolist(), open(f"{WORK}/preds_{name}.json", "w"))
    bench_record({"model": name, "accuracy": k / n, "ci_low": lo, "ci_high": hi,
                  "examples_per_s": n / elapsed, "latency_ms": 1000 * elapsed / n,
                  "peak_vram_gib": peak, "frac_pred_positive": float(np.mean(pred == 0)), "n_eval": n})
    print(f"   acc={k / n:.1%} [{lo:.1%}, {hi:.1%}] | {1000 * elapsed / n:.0f} ms/ex | pic VRAM {peak:.2f} Gio "
          f"| predit positive : {np.mean(pred == 0):.0%}")
    del model
    cleanup()

# Mistral sans adapter : dit si le QLoRA du teacher apporte quelque chose
run_benchmark("teacher_base", TEACHER_ID, TEACHER_PROMPT, TEACHER_LABELS, lambda: load_teacher_base().eval())
run_benchmark("teacher", TEACHER_ID, TEACHER_PROMPT, TEACHER_LABELS, load_teacher)
run_benchmark("student_raw", STUDENT_ID, STUDENT_PROMPT, STUDENT_LABELS, lambda: load_student_fp16())
if RUN_HARD_ABLATION:
    run_benchmark("student_hard", STUDENT_ID, STUDENT_PROMPT, STUDENT_LABELS,
                  lambda: load_student_fp16(f"{WORK}/student_hard_adapter"))
run_benchmark("student_distil", STUDENT_ID, STUDENT_PROMPT, STUDENT_LABELS,
              lambda: load_student_fp16(f"{WORK}/student_distil_adapter"))

In [ ]:
order = [m for m in ["teacher_base", "teacher", "student_raw", "student_hard", "student_distil"] if m in set(bench_df["model"])]
t = bench_df.set_index("model").loc[order].copy()

# Fidelite au teacher : % de predictions identiques sur le jeu d'eval
if os.path.exists(f"{WORK}/preds_teacher.json"):
    p_teacher = np.array(json.load(open(f"{WORK}/preds_teacher.json")))
    t["agree_teacher"] = [float(np.mean(np.array(json.load(open(f"{WORK}/preds_{m}.json"))) == p_teacher))
                          for m in t.index]

show = t.copy()
show["accuracy"] = [f"{a:.1%} [{lo:.1%}, {hi:.1%}]" for a, lo, hi in zip(t["accuracy"], t["ci_low"], t["ci_high"])]
cols = ["accuracy", "agree_teacher", "latency_ms", "examples_per_s", "peak_vram_gib", "frac_pred_positive"]
print(show[[c for c in cols if c in show.columns]].round(3).to_string())

if {"teacher", "student_distil"} <= set(t.index):
    print("\n--- Definition of Done (valeurs mesurees) ---")
    print(f"VRAM    : teacher {t.loc['teacher','peak_vram_gib']:.2f} Gio -> eleve {t.loc['student_distil','peak_vram_gib']:.2f} Gio "
          f"(x{t.loc['teacher','peak_vram_gib'] / t.loc['student_distil','peak_vram_gib']:.1f} moins)")
    print(f"Latence : teacher {t.loc['teacher','latency_ms']:.0f} ms -> eleve {t.loc['student_distil','latency_ms']:.0f} ms "
          f"(x{t.loc['teacher','latency_ms'] / t.loc['student_distil','latency_ms']:.1f} plus rapide)")
    print(f"Accuracy : teacher {t.loc['teacher','accuracy']:.1%} | eleve distille {t.loc['student_distil','accuracy']:.1%}")
    if "student_hard" in t.index:
        d = t.loc["student_distil", "accuracy"] - t.loc["student_hard", "accuracy"]
        print(f"Apport de la distillation vs labels durs : {d:+.1%} points")
        print("Les intervalles se chevauchent ? Alors l'ecart n'est PAS significatif : augmente N_EVAL (500-1000) avant de conclure.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = {"teacher_base": "#ADB5BD", "teacher": "#6C757D", "student_raw": "#FF6B6B", "student_hard": "#FFB454", "student_distil": "#00D4AA"}
x = list(t.index)

err = np.array([t["accuracy"] - t["ci_low"], t["ci_high"] - t["accuracy"]])
axes[0].bar(x, t["accuracy"], yerr=err, color=[colors[m] for m in x], capsize=4)
axes[0].set_title("Accuracy (95% CI)")
axes[0].set_ylim(0.4, 1.02)

axes[1].bar(x, t["latency_ms"], color=[colors[m] for m in x])
axes[1].set_title("Latency, batch 1 (ms)")

axes[2].bar(x, t["peak_vram_gib"], color=[colors[m] for m in x])
axes[2].set_title("Peak VRAM (GiB)")

for ax in axes:
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(f"{WORK}/distillation_benchmark.png", dpi=150)
plt.show()

## 8. Sauvegarde sur Hugging Face (optionnel)

Secret Kaggle `HF_TOKEN` requis (droits d'écriture). Les bases des adapters sont des identifiants du Hub (pas de chemin local),
donc pas de problème de métadonnées. L'adapter teacher ré-entraîné est poussé sous un **nouveau** nom : l'ancien reste dégénéré.

In [ ]:
PUSH = False   # les adapters sont recuperes par l'API Kaggle puis pousses depuis le PC

if PUSH:
    from huggingface_hub import HfApi, login
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("hf_write"))
    api = HfApi()

    jobs = [(f"{WORK}/student_distil_adapter", f"{HF_USER}/tinyllama-sentiment-distilled-from-mistral")]
    if TEACHER_MODE == "retrain":
        jobs.append((TEACHER_ADAPTER_DIR, f"{HF_USER}/mistral-nf4-sentiment-v2"))
    for path, repo in jobs:
        if os.path.exists(path):
            api.create_repo(repo, private=True, exist_ok=True)
            api.upload_folder(folder_path=path, repo_id=repo)
            print(f"Pousse -> https://huggingface.co/{repo}")
    api.upload_file(path_or_fileobj=BENCH_CSV, path_in_repo="benchmark.csv",
                    repo_id=f"{HF_USER}/tinyllama-sentiment-distilled-from-mistral")
else:
    print("PUSH = False : rien envoye.")